# 11_ensemble — OOF ensemble

Independent, leakage-controlled Kaggle phase. Exact image hashes define grouped OOF folds; test labels are never loaded.

In [ ]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
SUITE = Path.cwd()
while SUITE.name != "clean_breakthrough" and SUITE != SUITE.parent: SUITE = SUITE.parent
sys.path.insert(0, str(SUITE))
from src.dataset import discover_dataset, add_hashes
from src.validation import assign_folds, assert_no_overlap
from src.pipeline import run_experiment
from src.utils import seed_all
seed_all(42)
OUTPUT = Path("/kaggle/working/clean_breakthrough") if Path("/kaggle/working").exists() else SUITE / "runtime_outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA = discover_dataset()
TRAIN = add_hashes(DATA["train"], DATA["train_images"], verify=True)
TRAIN = assign_folds(TRAIN, n_splits=5, seed=42, hash_col="hash")
assert_no_overlap(TRAIN)
print("dataset", DATA["root"], "train/test", len(TRAIN), len(DATA["test"]))


In [ ]:
# Run dependable CPU members, then average their OOF predictions.
results=[]
for member in ("morphology", "sfs"):
    try:
        results.append(run_experiment(DATA, TRAIN, DATA["test"], member, OUTPUT, folds=5))
    except Exception as exc:
        print(member, "skipped:", exc)
if results:
    pred=np.mean([r["oof"] for r in results], axis=0)
    from src.metrics import summarize
    metrics=summarize(TRAIN.label.to_numpy(), pred)
    pd.DataFrame({"image_id":TRAIN.image_id,"prediction":pred,"experiment":"ensemble"}).to_csv(OUTPUT/"ensemble_oof.csv",index=False)
    (OUTPUT/"ensemble_metrics.json").write_text(json.dumps(metrics,indent=2))
    print("FINAL OOF RESULT", metrics)
else: print("No ensemble members available")